# 02. Demonstração dos baselines (LightGBM, XGBoost, CatBoost) por CV interna

Executa os três baselines no treino do fold externo 0 de um dataset da seleção, com validação cruzada interna estratificada de 3 folds, para verificar que carregamento, partições oficiais, treino e avaliação funcionam de ponta a ponta.

**Teste selado.** Este notebook pertence à fase de desenvolvimento: usa somente o TREINO de um fold externo oficial (`get_train_indices`) e mede o desempenho por validação cruzada INTERNA. O teste dos folds externos só é usado uma vez, em `python -m src.pipeline.final_test`, depois do congelamento (`python -m src.pipeline.freeze`).

In [ ]:
import sys
from pathlib import Path

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

import pandas as pd

from data.load_tabarena import get_train_indices, load_task, read_selection
from src.models.baselines import BASELINE_FACTORIES
from src.pipeline.cv_grid import inner_cv_splits, stratified_subsample
from src.pipeline.evaluate import fit_predict_evaluate

In [ ]:
task_id = read_selection()[0]
ds = load_task(task_id)
train_idx = get_train_indices(task_id, folds=[0])[0]   # somente o treino do fold externo 0
X_dev, y_dev = stratified_subsample(ds.X.iloc[train_idx], ds.y[train_idx], 10_000, seed=42)
splits = inner_cv_splits(y_dev, n_folds=3, seed=42)
print(f'Dataset: {ds.name} (n={ds.n_samples}, p={ds.n_features}, k={ds.n_classes}, regime={ds.regime})')
print(f'Treino do fold externo 0: {len(train_idx)} instâncias; CV interna com {len(y_dev)}')

In [ ]:
rows = []
for name, factory in BASELINE_FACTORIES.items():
    for inner, (tr, va) in enumerate(splits):
        m = fit_predict_evaluate(
            factory(seed=42), X_dev.iloc[tr], y_dev[tr], X_dev.iloc[va], y_dev[va]
        ).to_dict()
        rows.append({'model': name, 'inner_fold': inner, **m})
val = pd.DataFrame(rows)
val.groupby('model')[['auc_ovo', 'accuracy', 'cross_entropy', 'total_time_s']].agg(['mean', 'std'])

Esses números são estimativas de **validação interna**, não de teste. O experimento completo da fase de desenvolvimento é executado com `python -m src.pipeline.run_validation`, que grava `results/validation.csv` e `results/validation_train.csv`.